In [8]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [9]:
load_dotenv()

True

In [11]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash"
)

In [12]:
# create a state

class LLMState(TypedDict):

    question: str
    answer: str

In [13]:
def llm_qa(state: LLMState) -> LLMState:

    # extract the question from state
    question = state['question']

    # form a prompt
    prompt = f'Answer the following question {question}'

    # ask that question to the LLM
    answer = model.invoke(prompt).content

    # update the answer in the state
    state['answer'] = answer

    return state

In [14]:
# create our graph

graph = StateGraph(LLMState)

# add nodes
graph.add_node('llm_qa', llm_qa)

# add edges
graph.add_edge(START, 'llm_qa')
graph.add_edge('llm_qa', END)

# compile
workflow = graph.compile()

In [18]:
# execute

intial_state = {'question': 'How far is moon from the earth?'}

final_state = workflow.invoke(intial_state)

print(final_state['answer'])



[{'type': 'text', 'text': "On average, the Moon is about **384,400 kilometers (238,855 miles)** away from Earth. \n\nHowever, because the Moon's orbit around Earth is elliptical (egg-shaped) rather than a perfect circle, the distance changes throughout the month:\n\n* **Closest approach (Perigee):** About **363,300 km (225,623 miles)**\n* **Farthest point (Apogee):** About **405,500 km (251,966 miles)**\n\n**To put that distance into perspective:**\n* You could fit all 7 other planets in our solar system side-by-side in the space between Earth and the Moon, with room to spare.\n* Light traveling from the surface of the Moon takes about **1.3 seconds** to reach Earth.", 'extras': {'signature': 'EukTCuYTARFNMg+TC3viXxBqtSKzmrmc5Z0ezwACorgvrkJA+EKzH+tWQFSC+/zyMGRin8QKlrMqHJpJMe34XwplOjWss7Gbv6Vg1Yo273fEE3D6AtoR8+B0DTxAh82n41sfAo4anx94p0ZMscuQlUuXgGhChVcDVNX+4o6K/JFB05hupT8XVjAC6dBvGElvolmH5KJnt/52UYsWZBqevi6YL4tTFprCwwjIceBMjpDbwF3s+huh5Xf4y4VIYk3BT5GihQPHLMooaX51w0bc4+2Afm8F25dPkp0+6MlFj